In [ ]:
# sys.path manipulation to include parent directory
import sys
from pathlib import Path

parent_folder = str(Path.cwd().parent)
if parent_folder not in sys.path:
    sys.path.append(parent_folder)

# Importing necessary libraries and modules
from typing import List, Union
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.config import ApplicationConfig

# Import the custom visualization module
from src.data_plotter import DataPlotter

In [ ]:
raw_df = pd.read_csv(ApplicationConfig().data_path)
display(raw_df.head())

# 1. Dataset overview

Before diving into data analysis, it is important to understand and document the dataset’s variables. This step supports a clearer understanding of the problem and aids in defining the data schema when working with databases. Additionally, it guides the exploratory analysis by clarifying the meaning and interpretation of each variable.

Briefly, the dataset represents the monitoring of a Floating Production Storage and Offloading (FPSO) equipment unit across a sequence of operational cycles, with the goal of identifying and predicting equipment failures. It includes setup configurations, sensor readings such as temperature, pressure, and frequency, as well as a target variable indicating whether the equipment was in a failure state.

| Variable          | Type          | Data Type |Interpretation                                                                  |
| ----------------- | ------------- | --------- |------------------------------------------------------------------------------- |
| Cycle             | Temporal      | Numeric (int) |Represents the passage of time in operational cycles |
| Preset\_1         | Setup | Categorical (int) | Operational configuration parameter of the equipment                      |
| Preset\_2         | Setup | Categorical (int) |Operational configuration parameter of the equipment                                                         |
| Temperature       | Sensor        | Numeric (float) |Temperature measured in a given operation cycle  
| Pressure       | Sensor        | Numeric (float) |Pressure measured in a given operation cycle                       |
| Vibrations\_X/Y/Z | Sensor        | Numeric (float) |Vibration measurements along the three spatial axes in a given operation cycle                            |
| Frequency         | Sensor        | Numeric (float) |Frequency measured in a given operation cycle                             |
| Fail              | Target        | Binary (bool) |Indicates whether the equipment is in a failure state (True) or normal (False)         |


# 2. Data Cleaning

Inspecting the raw dataset, it is possible to verify that the columns data types are not properly defined.

In [ ]:
# Displaying the data types of the DataFrame
raw_df.dtypes

## 2.1 Data Types

The float numeric columns are being interpreted as text because they use commas as decimal separators. To ensure proper numerical analysis in Python, these commas should be replaced with dots, and the columns should be converted to float data types.

In [ ]:
# Function to convert object columns with commas to float
def parse_comma_decimal_series(series: pd.Series) -> pd.Series:
    return series.str.replace(',', '.').astype('float64')

# Function to get object columns with commas in their values
def find_comma_decimal_columns(df: pd.DataFrame) -> List[str]:
    """
    Returns a list of columns that contain commas in their values.
    """
    return [
        col for col in df.columns
        if df[col].dtype == 'object' and df[col].str.contains(',').any()
    ]

def df_convert_comma_decimals(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adjusts the data types of columns in the DataFrame.
    Converts object columns with commas to float.
    """
    obj_columns = find_comma_decimal_columns(df)
    for col in obj_columns:
        df[col] = parse_comma_decimal_series(df[col])
    return df

In [ ]:
obj_columns = find_comma_decimal_columns(raw_df)
print(f"Columns with commas: {obj_columns}")

In [ ]:
# Applying the conversion function to the DataFrame
df = df_convert_comma_decimals(raw_df)

In [ ]:
# New data types of the DataFrame
print(df.dtypes)
display(df.head())

## 2.2 Null values and index

In [ ]:
# Checking for null values in the DataFrame
df.isnull().sum()

No null values were found

In order to allow better handling of the time dimension on the dataset, the variable **Cycle** will be used as index.

In [ ]:
# Define Cycle as the index
df.set_index('Cycle', inplace=True)

With all data cleaning pendencies solved, it is now possible to analyze the general statistics of the dataset.

In [ ]:
# Export processed DataFrame to CSV
df.to_csv(ApplicationConfig().processed_data_path, index=True)

### Key Findings of Section 2: Data Cleaning

- **Data Types Correction:** Several numeric columns (e.g., Temperature, Pressure, VibrationX/Y/Z, Frequency) were initially read as text due to the use of commas as decimal separators. These were successfully converted to proper float types, enabling numerical analysis.
- **Null Values:** The dataset contains no missing values, ensuring completeness for analysis.
- **Indexing:** The 'Cycle' column was set as the DataFrame index, reflecting the temporal nature of the data and facilitating time-based operations.

## 3. Sensored data analysis

In this step, the main goal is to perform an initial examination of the sensored variables. This may reveal general patterns, potential anomalies, and relationships between the sensor measurements and failures in the FPSO equipment.

In [ ]:
sensor_columns = ['Temperature','Pressure', 'VibrationX', 'VibrationY','VibrationZ', 'Frequency']

# Displaying descriptive statistics for sensored data
df[sensor_columns].describe()

In [ ]:
# Plotting boxplots to see sensored data distributions
fig = DataPlotter().plot_multiple_boxplots(df, columns=sensor_columns)

The above data indicates the great presence of outliers in sensor readings, **Temperature** and **VibrationY** are the most relevant in this context.

Possibly, these outliers are related to failure events, where the equipment may suffer from overheat or vibration anomalies.

The correlation matrix below is provided to help understand how the variables are related.

In [ ]:
fig = DataPlotter().plot_correlation_heatmap(df[sensor_columns], figsize=(12, 10))

**Moderate Correlations:**
- VibrationY-VibrationZ: 0.51, makes sense because they are sensoring the same effect on the equipment
- VibrationX-VibrationZ: 0.51, makes sense because they are sensoring the same effect on the equipment
- VibrationY-Frequency: 0.47
- Pressure-VibrationX: 0.44

**Weak Correlations:**
- Temperature shows the weakest correlations with other variables (0.17-0.42), suggesting thermal events may have different root causes than mechanical issues

- VibrationX-Frequency: Nearly zero correlation (0.01)

**No strong correlations (>0.6)** indicates each sensor captures distinct aspects of equipment behavior

As this study deals with data over time, it is important to check how variables behave through the operation cycles.

In [ ]:
# Plotting time series for Temperature. Filtering the DataFrame to a specific range for clarity
fig = DataPlotter().plot_time_series(df.iloc[200:400], 'Temperature')

Given the high variability and noise in the time series, smoothing techniques can be useful to simplify visualizations and improve interpretability.

Moving average is a simple and direct option to reduce noise and highlight overall trends in the data.

In [ ]:
# Plotting time series for all variables. Filtering the DataFrame to a specific range for clarity
smooth_df = df.copy()
for col in sensor_columns:
    smooth_df[f'{col}_smooth'] = smooth_df[col].rolling(window=5, center=True).mean()
    DataPlotter().plot_time_series(smooth_df.iloc[200:400], f'{col}_smooth')

The above time series are plotting data from Cycles 200 to 400 for better visualization. Even though it doesn't represente the entire dataset, it gives valuable insights about how the sensored variables changes over time.

**Synchronized Spikes:** All variables show major peaks around cycles 230, 260–270, and 340–350, probably related to failure events.

**Variable-Specific Behaviors:**
- Temperature: Very sharp and short spikes
- Pressure and VibrationZ: High peaks with continuous cycles of increases and decreases
- VibrationX: Very high peaks, takes longer to go back to normal
- VibrationY: Volatile with big plateaus
- Frequency: Sequence of plateaus, but generally the same baseline

### Key Findings – Section 3: Sensor Data

**Outlier Patterns:**
- **Temperature** and **VibrationY** show the strongest outliers.
- These outliers likely signal overheating or abnormal vibration during failures.

**Correlation Analysis:**
- **Moderate correlations** between vibration sensors:
  - VibrationY–VibrationZ: 0.51  
  - VibrationX–VibrationZ: 0.51  
  - VibrationY–Frequency: 0.47  
  - Pressure–VibrationX: 0.44  

- **Temperature is weakly correlated** with other variables, suggesting thermal issues have different causes.

- **No strong correlations**: each sensor captures a different aspect of equipment behavior.

**Time Series Patterns:**
- **Synchronized peaks** aligned with failures.
- **Sensor-specific behaviors**:
  - **Temperature**: Sharp, brief spikes (thermal events)  
  - **Pressure & VibrationZ**: Long, fluctuating peaks  
  - **VibrationX**: High peaks with slow recovery  
  - **VibrationY**: Volatile with extended plateaus  
  - **Frequency**: Plateaus with stable baseline  

**Predictive Insights:**
- **Outliers** marking failure events.
- **All sensors matter**: Each one shows a different failure pattern.
- **Strong early warning signals** come from synchronized sensor spikes.

# 4. Setup parameters analysis

This section examines the distribution of configuration parameters (Preset_1 and Preset_2) and how their relate with sensored data.

In [ ]:
# Preset_1 and Preset_2 distributions
print(df['Preset_1'].value_counts(normalize=True)*100)
print(df['Preset_2'].value_counts(normalize=True)*100)

Considering the entire dataset, both variables exhibit a well-balanced distribution across their available values. Preset_1 has three unique values, each used in approximately one-third of the operation cycles. Similarly, Preset_2 has eight possible values, all of which appear with roughly equal frequency throughout the data collection period.

In [ ]:
fig = DataPlotter().plot_multiple_histograms(df, ['Preset_1','Preset_2'])

In addition to analyzing data distribution, it’s useful to examine how sensor readings vary across different **Preset_1** and **Preset_2** configurations.

This can be done using basic statistics grouped by each setup

In [ ]:
def create_stats_summary(df: pd.DataFrame, target_col: Union[str, List[str]] = 'Fail_index',
                         agg_funcs: List[str] = ['mean', 'max', 'median']) -> pd.DataFrame:
    """Creates a summary DataFrame with specified aggregation functions for each sensor column.
    Args:
        df (pd.DataFrame): DataFrame containing the data.
        target_col (Union[str, List[str]]): Target column name(s) for grouping.
        agg_funcs (List[str]): List of aggregation functions to apply.
    Returns:
        pd.DataFrame: DataFrame with aggregated statistics.
    """
    # Get float columns for sensor data
    sensored_cols = df.select_dtypes(include=['float64']).columns.tolist()

    # Create aggregation dictionary
    agg_dict = {col: agg_funcs for col in sensored_cols}
    
    # Handle single or multiple target columns for count
    if isinstance(target_col, str):
        agg_dict[target_col] = 'count'  # duration in operation cycles
    elif isinstance(target_col, list):
        # For multiple columns, use the first one for count
        agg_dict[target_col[0]] = 'count'
    
    # Group by target column(s) and aggregate
    result_df = df.groupby(target_col).agg(agg_dict)
    
    return result_df

In [ ]:
# Statistics summary grouped by Preset_1
stats_preset_1 = create_stats_summary(df, target_col='Preset_1', agg_funcs=['mean','max','std'])
display(stats_preset_1)

On **Preset_1** the three groups have similar averages, but in some variables the STD and maximum values differs signifcantly, indicatin setups with more variability.

**Preset_1=1**, for example, have higher STD and maximum value on **Temperature**, **VibrationY**, **VibrationZ** and **Frequency**.

In [ ]:
fig = DataPlotter().plot_multiple_boxplots(df, sensor_columns, 'Preset_1')

Although, based on the boxplots, there is no evident pattern on the behavio of sensored data during different setups of **Preset_1**. 

Besides that, the boxplot shows great amount of outliers on the majority of variables, which once again can be good indicators of failures.

In [ ]:
# Statistics summary grouped by Preset_2
stats_preset_2 = create_stats_summary(df, target_col='Preset_2', agg_funcs=['mean','max','std'])
display(stats_preset_2)

In [ ]:
fig = DataPlotter().plot_multiple_boxplots(df, sensor_columns, 'Preset_2')

Overall, there is **no clear or consistent pattern** linking **Preset_2** to major shifts in the sensor values.

This is suggesting that configuration alone does not strongly affect sensor readings. On the other hand, specific combinatios of **Preset_** and **Preset_2** could have this effect and must be investigated.

In [ ]:
stats_preset_1_2 = create_stats_summary(df, target_col=['Preset_1','Preset_2'], agg_funcs=['mean', 'max', 'std'])
display(stats_preset_1_2)

In [ ]:
# Displaying highest values for each sensor column in the stats DataFrame
for col in sensor_columns:
    print(f"Highest mean {col}:", stats_preset_1_2[(col),('mean')].idxmax())
    print(f"Highest max {col}:", stats_preset_1_2[(col),('max')].idxmax())
    print(f"Highest std {col}:", stats_preset_1_2[(col),('std')].idxmax())
    print('\n')

**Preset_1 = 1**  
  Consistently linked to **high values across most metrics**, especially:
  - Temperature: **mean**, **max**, **std**
  - Pressure: **mean**, **max**, **std**
  - VibrationY  **mean**, **max**, **std**

**Preset_1 = 1**, combined with **Preset_2 = 4, 7, or 8**, is associated with the most intense sensor behavior —  suggesting these configurations may push the equipment closer to operational limits or failure conditions.


### Key Findings of Section 4: Setup Data Analysis

**Overall Distribution:**
- Both Preset_1 and Preset_2 show balanced distributions across all values during normal operations

**Individual Risk Levels:**
- **Preset_1=1**: Higher sensor variability and maximum values (riskier configuration)

- **Combination effects are significant** - specific preset pairs create notably different sensor behaviors
- Preset_1=1 consistently linked to higher Temperature, Pressure, and VibrationY readings

**Critical Combinations:**
- **High-Risk**: Preset_1=1 + Preset_2=4/7/8 show the most intense sensor behavior

**Operational Insight:**
- Configuration combinations matter more than individual settings
- Preset_1=1 combinations should be monitored more closely during critical operations

Now, with a better understanding of the sensor variables and equipment configurations, we can analyze the distribution of failure events and their relationship with the dataset variables.

This analysis will help interpret the causes of these events and improve the chances of accurately predicting them.

# 5. Failure events analysis

## 5.1 Failure distribution

In [ ]:
print(df['Fail'].value_counts())
print('\n')
print(df['Fail'].value_counts(normalize=True)*100)

fig = DataPlotter(figsize=(8,5)).plot_histogram(df, 'Fail')

According to the data, it is possible to affirm that the equipment was in a failure state for approximately **8.25%** of the time or **66** operation cycles.

As expected, the target variable is imbalanced, since equipment failure is a relatively rare event. However, the failure state is precisely what needs to be predicted. For this reason, it is essential to understand the equipment’s behavior during failure, despite the smaller amount of available data.

In [ ]:
# Simple bar plot showing failure events over time
fig, ax = plt.subplots(figsize=(12, 6))

# Create a simple time series showing failure state
failure_over_time = df['Fail'].astype(int)  # Convert boolean to 0/1

# Plot as bar chart
ax.bar(failure_over_time.index, failure_over_time, 
       color=['red' if x == 1 else 'lightblue' for x in failure_over_time], 
       alpha=0.7, width=1)

ax.set_xlabel('Operation Cycle')
ax.set_ylabel('Failure State')
ax.set_title('Failure Events Over Time')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Normal', 'Failure'])

# Add grid for better readability
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

When examining these events over a timeline, the intervals between failures appear consistent, suggesting possible seasonal patterns. Additionally, the data shows an increase in the duration of failure events throughout the analyzed period.

### Key Findings of Section 5.1. Failure distribution

- **Failure Events:** The equipment was in fail state for 66 of 800 operation cycles (~8.25% of total time).

- Seasonal pattern on occurence of failures, but recent trend of longer durations.

## 5.2 Behavior of sensored variables during failures

First, the numeric variables, such as Temperature and Pressure, will be analyzed. This section aims to identify significant differences in their values between normal and failure states.

In [ ]:
# Calculate statistics by failure state
stats_by_failure_state = create_stats_summary(df, target_col='Fail', agg_funcs=['mean', 'median', 'std']).drop(columns=['Fail'])
display(stats_by_failure_state)

In [ ]:
stats_by_failure_state = create_stats_summary(df, target_col='Fail', agg_funcs=['mean', 'median','max', 'min', 'std']).drop(columns=['Fail'])

All five sensor-based variables exhibit higher descriptive statistics during the failure state. The boxplots below visually illustrate this finding. Below, a series of boxplots represent this find visually.

In [ ]:
# Boxplot for each numeric column grouped by 'Fail'
# This will help visualize the distribution of each feature for both failure states
fig = DataPlotter().plot_multiple_boxplots(df, df.select_dtypes(float).columns.tolist(), 'Fail')

It becomes evident that all numeric variables tend to be higher when the equipment is in a failure state. This suggests that increasing values in these variables during operation cycles may be indicators of a possible future equipment failure. Additionally, the distributions of all variables show minimal overlap between normal and failure states, indicating strong predictive potential.

Among them, **VibrationY** and **Pressure** demonstrate the most significant differences between normal and failure conditions. This shows their potential importance in failure detection and suggests they may play a critical role in the feature importance analysis of the machine learning model to be developed.

One simple way to quantify these difference is calculating the mean differences of each variable in normal and failure state.

In [ ]:
def mean_difference_by_target(df: pd.DataFrame, columns: List[str], target: str = 'Fail') -> pd.Series:
    """
    Calculates the mean difference for the specified columns between True and False states of the target variable.

    Args:
        df (pd.DataFrame): The dataframe containing the data.
        columns (list): List of column names to calculate mean difference for.
        target (str): The binary target column name.

    Returns:
        pd.Series: Mean difference (True - False) for each column.
    """
    assert target in df.columns, f"Target column '{target}' not found in DataFrame."
    assert all(col in df.columns for col in columns), "One or more specified columns not found in DataFrame."
    
    means_true = df[df[target] == True][columns].mean()
    means_false = df[df[target] == False][columns].mean()

    return means_true - means_false

In [ ]:
# Calculate mean differences for float columns by target and sort results
mean_difference_by_target(df, df.select_dtypes(float).columns.tolist(), 'Fail').sort_values(ascending=False)

Once again, the data reveals a significant increase in sensored variables during equipment failures. The mean **VibrationY** is approximately 54 units higher and mean **Pressure** is 40 units higher when the equipment is in a failure state.

### Key Findings of Section 5.2: Behavior of sensored variables during failures

- **Sensor Elevation**: All five sensor variables exhibit consistently higher values during failure states.

- **Primary Failure Indicators**: **VibrationY** and **Pressure** show the most significant mean differences between normal and failure states:
  - **VibrationY**: ~54 units higher during failures
  - **Pressure**: ~40 units higher during failures
  - These variables demonstrate strong predictive potential for failure detection

- **Clear Distribution Separation**: Boxplot analysis reveals minimal overlap between normal and failure state distributions across all sensor variables, suggesting strong discriminative power for machine learning models.

- **Early Warning Potential**: The consistent elevation of sensor readings during failures suggests these variables can serve as reliable indicators for developing predictive maintenance strategies and early warning systems.

## 5.3 Distribution of setup parameters during failures

The setup variables Preset_1 and Preset_2 define how the equipment is configured during an operation cycle. Analyzing their relationship with failure occurrences is essential to identify configuration combinations that are more likely to result in failures.

When isolating the operation cycles with failure, certain setup parameters appear more frequently, such as values 1 and 2 for **Preset_1**, and 1 and 5 for **Preset_2**. Conversely, parameter 4 on **Preset_2** is relatively uncommon in failure events.

In [ ]:
# Preset_1 and Preset_2 distributions considering only failure events
print(df.query('Fail == True')['Preset_1'].value_counts(normalize=True)*100)
print(df.query('Fail == True')['Preset_2'].value_counts(normalize=True)*100)

In [ ]:
# Plotting histograms for Preset_1 and Preset_2 for failure events
fig = DataPlotter().plot_multiple_histograms(df.query("Fail == True"), ['Preset_1','Preset_2'])

Taking it a step further, specific combinations of **Preset_1** and **Preset_2** configurations may be correlated with failure events. To visualize this relationship, a heatmap can be used to show how the proportion of failures varies across different equipment setup combinations.

In [ ]:
# Grouping by Preset_1 and calculating failure rate and number of cycles with failure
print(df.groupby(['Preset_1']).agg({'Fail': ['mean', 'sum']}).rename(columns={'mean': 'Failure Rate', 'sum': 'Cycles with Failure'}))
print('\n')
# Grouping by Preset_2 and calculating failure rate and number of cycles with failure
print(df.groupby(['Preset_2']).agg({'Fail': ['mean', 'sum']}).rename(columns={'mean': 'Failure Rate', 'sum': 'Cycles with Failure'}))

**Preset_1**
- Preset_1=1 has a higher failure rate (~10%) in comparison with the other two options.

**Preset_2**
- Preset_2=5 and Preset_2=1 appears to be risky, both with approximately 12% of failure rate.
- Preset_2=4 could be considered the safest setting, with only 3% of failure rate. 

In [ ]:
# Heatmap plotting failure rates by Preset_1 and Preset_2
fig = DataPlotter().plot_failure_rate_heatmap(df, row_var='Preset_1', col_var='Preset_2', target_var='Fail')

The heatmap reveals that certain equipment setups stand out with a noticeably higher occurrence of failures.

**Configurations with high failure rates**:
- Preset_1=1, Preset_2=5: Highest failure rate at 16%, almost double of the mean failure rate (8.25%).
- Preset_1=3, Preset_2=5: Second highest at 14%
- Preset_1=1, Preset_2=2: Third highest at 13%

**Configurations with low failure rates**:
- Preset_1=3, Preset_2=4: Zero failures (safest combination)
- Preset_1=2, Preset_2=4: Only 3% failure rate

Without specific knowledge of how the **Preset_1** and **Preset_2** parameters control equipment operation, it is difficult to recommend direct actions to minimize failure risk on the FPSO.

Nonetheless, based on the analyzed data, it is advisable to avoid configurations associated with high failure rates, especially during critical operations or peak production periods. As a safer alternative, combinations such as Preset_1 = 3 and Preset_2 = 4, which showed lower failure occurrences, could be prioritized when possible.

These critical equipment configurations should be closely monitored alongside sensor-based variables. The combination of high-risk setups with increases in variables such as vibration and pressure may serve as early warning signals of equipment failure.

### Key Findings of Section 5.3: Distribution of setup parameters during failures

- **Balanced Overall Distribution**: Both Preset_1 (3 values) and Preset_2 (8 values) show well-balanced distributions, with each configuration used approximately equally during operations.

- **High-Risk Individual Configurations**
  - **Preset_1=1**: Higher failure rate (~10%) compared to other Preset_1 options
  - **Preset_2=5** and **Preset_2=1**: Both show risky behavior with ~12% failure rates
  - **Preset_2=4**: Safest individual setting with only 3% failure rate

- **Critical Configuration Combinations**:
  - **Preset_1=1, Preset_2=5**: Highest failure rate at 16% (nearly double the overall 8.25% rate)
  - **Preset_1=3, Preset_2=5**: Second highest at 14%
  - **Preset_1=1, Preset_2=2**: Third highest at 13%

- **Safe Configuration Combinations**:
  - **Preset_1=3, Preset_2=4**: Zero failures recorded (safest combination)
  - **Preset_1=2, Preset_2=4**: Only 3% failure rate

- **Operational Recommendations**: Avoid high-risk configurations (especially Preset_1=1 with Preset_2=5) during critical operations.

- **Early Warning Integration**: High-risk equipment configurations could be monitored alongside sensor variables, as the combination of risky setups with elevated vibration and pressure readings may serve as compound early warning signals for equipment failure.

## 5.4 Caracterizing failure events

Another important aspect of this study is identifying distinct failure events and analyzing them in terms of their duration, causes, and associated variable behavior.

The first step is to find these events and categorize them on the dataset.

In [ ]:
def find_failure_transitions(fail_series: pd.Series) -> dict:
    """
    Finds all failure transitions (starts and stops) in a boolean Series.

    Args:
        fail_series (pd.Series): A pandas Series of boolean values indicating failure state over time.

    Returns:
        dict: Dictionary containing:
            - 'starts': List of cycle ids where transitions from False to True occurred
            - 'stops': List of cycle ids where transitions from True to False occurred
            - 'events': List of tuples (start, stop) for each failure event
    """
    # Check if the input series is of boolean dtype
    if fail_series.dtype != bool:
        raise ValueError("Input series must be of boolean dtype.")
    
    # Use shift to 'move' rows one position down and verify previous state
    previous_state = fail_series.shift(1, fill_value=False)
    
    # Find transitions from False to True (failure starts)
    transition_starts = (~previous_state) & fail_series
    starts = list(transition_starts[transition_starts == True].index)
    
    # Find transitions from True to False (failure stops)
    transition_stops = previous_state & (~fail_series)
    stops = list(transition_stops[transition_stops == True].index)
    
    # If series ends with failure, add the last index as stop
    if len(starts) > len(stops):
        stops.append(fail_series.index[-1] + 1)  # Add one to include the last cycle as stop
    
    # Create failure events as (start, stop) tuples
    events = list(zip(starts, stops))
    
    return {
        'starts': starts,
        'stops': stops,
        'events': events
    }

In [ ]:
# Finding failure transitions in the 'Fail' column
failure_events = find_failure_transitions(df['Fail'])
display(failure_events)

In [ ]:
n = len(failure_events['events'])
failure_duration = df['Fail'].sum()

print(f"Total duration of failure state: {failure_duration} cycles")
print(f"Total number of failure events: {n}")
print(f"Mean failure state duration per occurence: {failure_duration / n} cycles")

Therefore, the data reveals a total of **10 failure events** throughout the time series, resulting in **66 operation cycles** in the failure state.

These events are detailed below to provide a clearer understanding of each individual occurrence.

In [ ]:
def label_failures(df: pd.DataFrame, failure_events: dict, fail_index_column: str = 'Fail_index') -> pd.DataFrame:
    """
    Labels the failure periods in the DataFrame with a unique index for each failure.

    Args:
        df (pd.DataFrame): DataFrame containing the failure data with 'Fail' column.
        failure_events (dict): Dictionary containing failure events with 'events' key.
        fail_index_column (str): Name of the column to store failure indices.

    Returns:
        pd.DataFrame: DataFrame with an additional 'Fail_index' column indicating failure periods.
    """
    # Initialize a new column for failure indices
    df[fail_index_column] = 0

    # Loop through each failure period and assign an index
    for i, event in enumerate(failure_events['events']):
        start, stop = event
        print(f"Failure {i+1} start at operation cycle: {start}")
        print(f"Failure {i+1} stop at operation cycle: {stop}")

        # Assigning a failure index to the DataFrame for each failure period
        mask = (df.index >= start) & (df.index < stop)
        df.loc[mask, 'Fail_index'] = i + 1

    return df

In [ ]:
# Applying the function to label failures in the DataFrame
df = label_failures(df, failure_events=failure_events, fail_index_column='Fail_index')

In [ ]:
# Sample of the DataFrame with failure indices indicating different failure events
df.head(15)

With the new label added to the dataset, it is now possible to quantify the duration of each event. Longer events may indicate more critical or severe failures in the equipment.

In [ ]:
from matplotlib.ticker import MaxNLocator
from matplotlib import pyplot as plt

# Plotting the number of cycles per failure event
failures = df.query('Fail == True').groupby('Fail_index').size()

print(f"Median number of cycles per failure event: {failures.median()}")

ax = failures.plot(kind='bar', figsize=(12, 6), title='Number of cycles per failure event')
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_ylabel('Number of cycles')
ax.set_xlabel('Failure event id')

plt.show()

It is now evident that failure events 6 through 10 lasted significantly longer than the first five. Notably, **Event 6** exceeded the average failure duration by nearly three times and the median by six times.

This trend of increasingly prolonged failure events over time may indicate chronic issues with the FPSO equipment or operational challenges. It represents a critical concern that must be addressed to minimize the impact of extended downtime on oil extraction.

The following analysis will look at these events in greater detail to better understand their characteristics and potential root causes.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Stacked barplot for Preset_1 configurations per failure event
df.query('Fail==True').groupby(['Fail_index', 'Preset_1']).size().unstack().plot(
    kind='bar',
    stacked=True,
    colormap='Paired',
    ax=ax
)

# Adjust axes and labels
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_ylabel('Number of cycles')
ax.set_xlabel('Failure event id')
ax.set_title('Preset_1 Configurations per Failure Event')

plt.tight_layout()
plt.show()

The barplot reveals important operational patterns during individual failure events:

**Configuration Switching During Failures:**
- Multiple Preset_1 settings are actively used within individual failure events, indicating operators systematically attempt different configurations to restore normal operation
- This configuration switching behavior is most pronounced in longer failure events (6-10)

**Preset_1 Usage Patterns:**
- **Preset_1=1**: Predominant in the longest failure event (Event 6);
- **Preset_1=2**: Appeared frequently in major critical failures (Events 7 and 9);
- **Preset_1=3**: Least utilized across nearly all failure events.

This pattern indicates that operational strategy becomes more aggressive and varied as failures persist, with operators cycling through available configurations in an attempt to restore normal state.

In [ ]:
# Create a larger figure
fig, ax = plt.subplots(figsize=(10, 5))

# Stacked barplot for Preset_2 configurations per failure event
df.query('Fail==True').groupby(['Fail_index', 'Preset_2']).size().unstack().plot(
    kind='bar',
    stacked=True,
    colormap='Paired',
    ax=ax
)

# Adjust axes and labels
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_ylabel('Number of cycles')
ax.set_xlabel('Failure event id')
ax.set_title('Preset_2 Configurations per Failure Event')

plt.tight_layout()
plt.show()

When analyzing **Preset_2**, similar behavior can be found.

**Configuration Diversity During Failures:**
- Multiple Preset_2 values are actively used within individual failure events, indicating operators are systematically trying different configurations to restore normal operation
- This configuration switching behavior is most pronounced in longer failure events (6-10)

**Temporal Pattern in Later Failures:**
- **Preset_2=7 and Preset_2=6** appears consistently across nearly all extended failure periods;
- **Preset_2=5** is used in 60% of all events, but usually for few operation cycles each time.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Create a pivot table showing combinations for each failure event
combo_heatmap = df.query('Fail == True').reset_index().pivot_table(
    values='Cycle', 
    index='Fail_index', 
    columns=['Preset_1', 'Preset_2'], 
    aggfunc='count', 
    fill_value=0
)

sns.heatmap(combo_heatmap, annot=True, cmap='Reds', fmt='d')
plt.title('Preset Combinations per Failure Event')
plt.show()

When analyzing Preset combinations (Preset_1–Preset_2), similar patterns emerge.

**Configuration During Failures:**
- Multiple Preset combinations are used within individual failure events, reflecting a effort to adjust configurations in response to equipment issues;
- This switching behavior is more evident in longer failure events (6–10), where operators or automated controls tried many different combinations;
- Certain configurations, such as Preset_1=1 with Preset_2=7 and Preset_1=1 with Preset_2=6, become more prominent during extended failures, especially in events 6–10.

Besides the equipment configuration, it is also important to examine the sensor data behavior during the failure events. This will help reveal how failures manifest in these variables and may allow the identification of different root causes.

In [ ]:
def plot_failure_summary_heatmap(df: pd.DataFrame, target_col: str = 'Fail_index', agg_funcs: List[str] = ['mean', 'max', 'median'],
                                figsize: tuple = (16, 10),
                                title: str = "Failure Events Summary - Sensor Statistics") -> None:
    """    Creates a heatmap of failure summary statistics with separate scaling for each variable.
    Args:
        df (pd.DataFrame): DataFrame with failure statistics by event
        target_col (str): Target column name for grouping.  
        agg_funcs (List[str]): List of aggregation functions to apply.
        figsize (tuple): Figure size
        title (str): Plot title
    """
    failures_summary = create_stats_summary(df, target_col=target_col, agg_funcs=agg_funcs)

    # Flatten the multi-level columns for easier processing
    df_flat = failures_summary.copy()
    df_flat.columns = [f"{col[0]}_{col[1]}" for col in df_flat.columns]
    
    # Separate sensor variables from duration
    sensor_cols = [col for col in df_flat.columns if col != f"{target_col}_count"]

    # Create figure with subplots
    fig, axes = plt.subplots(1, 2, figsize=figsize, gridspec_kw={'width_ratios': [4, 1]})
    
    # Plot 1: Sensor variables heatmap (normalized by column)
    sensor_data = df_flat[sensor_cols]
    sensor_normalized = sensor_data.apply(lambda x: (x - x.min()) / (x.max() - x.min()), axis=0)
    
    sns.heatmap(sensor_normalized, 
                annot=sensor_data.round(1),  # Show actual values
                cmap='RdYlBu_r', 
                cbar_kws={'label': 'Normalized Scale (0-1)'}, 
                ax=axes[0],
                fmt='g')
    
    axes[0].set_title('Sensor Variables (Each Column Independently Scaled)')
    axes[0].set_ylabel('Failure Event ID')
    
    # Plot 2: Duration heatmap
    duration_data = df_flat[[f"{target_col}_count"]]
    duration_normalized = (duration_data - duration_data.min()) / (duration_data.max() - duration_data.min())
    
    sns.heatmap(duration_normalized,
                annot=duration_data,
                cmap='Reds',
                cbar_kws={'label': 'Duration Scale'},
                ax=axes[1],
                fmt='g')
    
    axes[1].set_title('Duration\n(Cycles)')
    axes[1].set_ylabel('')
    
    plt.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Create heatmap visualization
failure_summary = create_stats_summary(df.query("Fail == True"), target_col='Fail_index', agg_funcs=['mean', 'max', 'std'])
display(failure_summary)

# Create heatmap visualization
plot_failure_summary_heatmap(df.query("Fail == True"), 
                            figsize=(26, 13),
                            agg_funcs=['mean', 'max', 'std'],
                            title="Failure Events - Sensor Statistics Heatmap")

Interesting insights can be drawn from the general statistics of sensor data across each failure event.

**Short failures (Events 1-5):**
- Last up to 3 operation cycles.
- Marked by sharp peaks in temperature and vibration, usually with low variability (STD).
- Events 3, 4, and 5 show notably high Temperature, with Event 3 having very high variability (STD ~30), suggesting instability.
- High VibrationX in Event 1 and VibrationZ in Event 5, both with low variability — indicating brief but intense conditions.
- Event 3 stands out with high values in most variables and high variability in Temperature and Pressure — likely an outlier.
- General pattern: Sudden, short-lived failures. Event 3 may indicate a different failure mechanism.

**Long failures (Events 6-10):**
Last 5 or more cycles.
- Characterized by consistently high values in vibration and pressure, with generally higher variability;
- Event 6 shows extreme variability (e.g., VibrationX STD ~43), suggesting mechanical instability;
- Event 7 has the highest Temperature, but moderate variability;
- Event 10 combines high Temperature and VibrationZ variability;
- Events 8 and 9 have high readings but lower variability;
- General pattern: Long and chronic failures, with some events being much more unstable (6 and 10).


One possible interpretation is that during short failures, the equipment experienced sudden peaks in sensor readings that quickly returned to normal levels. In longer failures, these variables sustained abnormal levels over time, reinforcing the hypothesis of potentially chronic mechanical or process issues. This may also indicate a progressive deterioration in equipment condition over the analyzed period.

Clustering can be useful in this context to validate the above grouping of similar failure events based on sensor behavior.

In [ ]:
# Cluster failure events based on sensor characteristics
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Standardize and cluster
scaler = StandardScaler()
scaled_features = scaler.fit_transform(failure_summary.fillna(0))

kmeans = KMeans(n_clusters=2, random_state=42)
failure_clusters = kmeans.fit_predict(scaled_features)

failure_summary['Cluster'] = failure_clusters
clusters = {}

print("Failure Event Clusters:")
for i, cluster in enumerate(failure_clusters):
    print(f"Event {i+1}: Cluster {cluster}")
    clusters[i+1] = cluster

The two clusters align closely with the initial interpretation of short and long failure periods. The only difference is the reassignment of Event 3 and Event 8, now labeled as Cluster 1 and Cluster 0, respectively. This reassignment is reasonable, as Event 3 showed unusually high variability for a short failure, resembling long failures more closely.

To better highlight these differences, it is helpful to recompute a summary of descriptive statistics by cluster.

In [ ]:
# Addition of cluster column on the main DataFrame
df['Clusters'] = df['Fail_index'].map(clusters)

create_stats_summary(df.query('Fail == True'),
                     target_col='Clusters',
                     agg_funcs=['mean', 'max', 'std'])

The data reveals that Cluster 1, which includes most long-duration failures, exhibits significantly higher values and variability in key variables like **Pressure** and **VibrationY**. Failures in this cluster also show the highest maximum readings across all sensor variables. In contrast, **Frequency** and **VibrationZ** display more consistent behavior across both clusters, suggesting these variables are less discriminative for failure type

### Key Findings of Section 5.4: Caracterizing failure events

**Count of Failure Events:** The equipment had 10 failure events during the period of 800 cycles.

**Failure Durations:**
- **Short failures** (Events 1, 2, 4, 5): Lasted 1–3 cycles, with quick spikes and recoveries
- **Long failures** (Events 6–10): Lasted 5+ cycles, showing sustained issues
- Failures became longer over time, pointing to progressive degradation

**Preset configurations during failures:**
- **Configuration Switching Strategy**: Multiple preset combinations are used within individual failure events, indicating operators systematically attempt different configurations to restore normal operation;
- **Escalating Complexity**: Configuration switching behavior becomes most pronounced in longer failure events (6-10);
- **Preset_1 Patterns**:
  - **Preset_1=1**: Predominant in the longest failure event (Event 6) and critical shorter events
  - **Preset_1=2**: Frequently used in major critical failures (Events 7 and 9)
  - **Preset_1=3**: Least utilized across nearly all failure events
- **Preset_2 Temporal Evolution**: 
  - **Preset_2=6 and Preset_2=7** become dominant in later failure events (5-10)

**Sensor Behavior:**
- **VibrationY** and **Pressure** had the most noticeable increases during failures
- **Temperature** peaked in Events 3, 4, 5, and 7
- **Variability (STD)** revealed two patterns:
  - **Low STD:** Stable spikes (acute issues)
  - **High STD:** Erratic behavior (unstable failures)

**Failure Clusters:**
- **Cluster 0** (Events 1, 2, 4, 5, 8): Stable failures, low variability
- **Cluster 1** (Events 3, 6, 7, 9, 10): Unstable failures, high variability

**Notable Events:**
- **Event 3:** Short but highly unstable – reclassified correctly due to extreme STD
- **Event 6:** Longer and most unstable – VibrationX STD ~43
- **Event 8:** A long but stable failure, possibly with a different root cause

**Conclusion:** Failures are becoming more frequent and unstable, reinforcing the need for preventive maintenance to avoid critical breakdowns.

## 5.5 Pre-failure event analysis

Besides analyzing equipment behavior during failures, it is also important to understand how the sensor variables behaved in the moments leading up to these events. To achieve this, a good approach is to analyze failure events with buffers of operation cycles preceding each failure onset. This can help identify early indicators or gradual trends that precede equipment failures.

In [ ]:
def get_buffer_around_index(df: pd.DataFrame, 
                           start_index: int, 
                           stop_index: int,
                           buffer_before: int = 5,
                           buffer_after: int = 5,
                           exclude_failure_cycles: bool = False) -> pd.DataFrame:
    """
    Extracts cycles before and after a specific index for analysis of patterns around that point.
    
    Args:
        df (pd.DataFrame): DataFrame with the data
        start_index (int): The start index around which to create the buffer
        stop_index (int): The stop index around which to create the buffer
        buffer_before (int): Number of cycles before the start index to include
        buffer_after (int): Number of cycles after the stop index to include
        exclude_failure_cycles (bool): If True, excludes cycles that are already in failure
        
    Returns:
        pd.DataFrame: DataFrame containing only the cycles in the buffer around target index
    """
    buffer_indices = []
    
    # Calculate the range of the buffer
    start_buffer = start_index - buffer_before
    end_buffer = stop_index + buffer_after-1
    
    # Ensure we don't go beyond the dataset boundaries
    start_buffer = max(start_buffer, df.index.min())
    end_buffer = min(end_buffer, df.index.max())
    
    # Add indices to buffer if they exist in the dataset
    for idx in range(start_buffer, end_buffer + 1):
        if idx in df.index:
            # If exclude_failure_cycles=True, only add if not in failure
            if exclude_failure_cycles:
                if not df.loc[idx, 'Fail']:
                    buffer_indices.append(idx)
            else:
                buffer_indices.append(idx)
    
    # Remove duplicates and sort
    buffer_indices = sorted(list(set(buffer_indices)))
    
    # Return the subset of DataFrame
    return df.loc[buffer_indices].copy()

In [ ]:
for event in failure_events['events']:
    print(f"Failure event from cycle {event[0]} to {event[1]}")
    buffer = get_buffer_around_index(df,
                                    start_index=event[0],
                                    stop_index=event[0],
                                    buffer_before=3,
                                    buffer_after=1,
                                    exclude_failure_cycles=False)

    display(buffer)
    display(buffer.select_dtypes(float).std())
    print('\n')

This preliminary analysis of the operation cycles preceding failures shows a consistent increase in sensor readings leading up to the failure events. In nearly all 10 recorded failures, at least one variable exhibited a sharp rise or significant variation shortly before the failure occurred.

Therefore, using rolling windows to create new features is a promising approach to predict equipment failures and uncover distinct patterns among different failure types.

In [ ]:
def calculate_pre_failure_rolling_features(df: pd.DataFrame, 
                                         failure_events: dict,
                                         buffer_size: int = 4,
                                         windows: List[int] = [3],
                                         sensor_cols: List[str] = None) -> pd.DataFrame:
    """
    Calculate rolling window features for cycles preceding each failure event.
    Uses existing get_buffer_around_index function.
    
    Args:
        df (pd.DataFrame): Main DataFrame with sensor data
        failure_events (dict): Dictionary with failure events (from find_failure_transitions)
        buffer_size (int): Number of cycles before failure to analyze
        windows (List[int]): Window sizes for rolling calculations
        sensor_cols (List[str]): Sensor columns to analyze. If None, uses float columns
        
    Returns:
        pd.DataFrame: DataFrame with pre-failure rolling features for each event
    """
    if sensor_cols is None:
        sensor_cols = df.select_dtypes(include=['float64']).columns.tolist()
    
    pre_failure_features = []
    
    # Iterate through each failure event
    for event_idx, (start_cycle, _) in enumerate(failure_events['events']):
        # Use existing function to get buffer
        buffer_data = get_buffer_around_index(
            df=df,
            start_index=start_cycle,
            stop_index=start_cycle,  # Only need start point
            buffer_before=buffer_size,
            buffer_after=1,  # buffer = 1 to consider failure start point
            exclude_failure_cycles=False  # Exclude cycles already in failure
        )
        
        if len(buffer_data) > 0:
            # Calculate rolling features for this buffer
            event_features = {'Fail_index': event_idx + 1}
            
            for window in windows:
                for col in sensor_cols:
                    if col in buffer_data.columns:
                        # Rolling statistics, get value from the beginning of the failure
                        rolling_mean = buffer_data[col].rolling(window=window, min_periods=1).mean().iloc[-1]
                        rolling_std = buffer_data[col].rolling(window=window, min_periods=1).std().iloc[-1]
                        
                        # Calculate percent change
                        # If buffer is smaller than window, set pct_change to 0
                        if len(buffer_data) > window:
                            pct_change = buffer_data[col].pct_change(periods=window).iloc[-1]
                        else:
                            pct_change = 0
                        
                        # Store features
                        event_features[f'{col}_rolling_mean_{window}'] = rolling_mean
                        event_features[f'{col}_rolling_std_{window}'] = rolling_std
                        event_features[f'{col}_pct_change_{window}'] = pct_change
            
            pre_failure_features.append(event_features)
    
    return pd.DataFrame(pre_failure_features)

# Usage:
pre_failure_df = calculate_pre_failure_rolling_features(
    df=df,
    failure_events=failure_events,
    buffer_size=2,
    windows=[3],
    sensor_cols=['Temperature', 'Pressure', 'VibrationX', 'VibrationY', 'VibrationZ', 'Frequency']
)

print(f"Pre-failure features shape: {pre_failure_df.shape}")
display(pre_failure_df)

In [ ]:
# Standardize and cluster
scaler = StandardScaler()
scaled_features = scaler.fit_transform(pre_failure_df.fillna(0))

kmeans = KMeans(n_clusters=2, random_state=42)
pre_failure_clusters = kmeans.fit_predict(scaled_features)

pre_failure_df['Cluster'] = pre_failure_clusters
clusters = {}

print("Failure Event Clusters:")
for i, cluster in enumerate(pre_failure_clusters):
    print(f"Event {i+1}: Cluster {cluster}")
    clusters[i+1] = cluster

In [ ]:
# Dataframe with pre-failure rolling features and clusters
display(pre_failure_df)

This clustering approach focuses on identifying different root causes of failures by analyzing how sensored variables behave in the cycles leading up to failure events. Based on this, two clusters were formed:

- Cluster 0: Events 2, 4, 10
- Cluster 1: Events 1, 3, 5, 6, 7, 8, 9

This suggests the presence of two distinct pre-failure behavior patterns, which may reflect different failure mechanisms or root causes across events.

In [ ]:
# Statistics summary for pre-failure features by cluster
create_stats_summary(pre_failure_df, target_col='Cluster', agg_funcs=['mean'])

Clear differences can be found between those clusters:
- Temperature: Higher mean and variability in Cluster 0 — suggesting sharp spikes before failure. In Cluster 1, values are lower and more stable.
- Pressure: Higher and more variable in Cluster 1 — indicating buildup over time.
- Vibrations: Stronger and more consistent in Cluster 1; more erratic in Cluster 0.
- Frequency: Slightly more variation in Cluster 0; overall similar.

Understanding these distinct pre-failure signatures could allow monitoring approaches:

- **Cluster 0 (Temperature-Driven Failures)**: Characterized by sudden temperature spikes with high variability before failure. These events require immediate alerts when temperature volatility exceeds normal ranges.

- **Cluster 1 (Pressure/Vibration-Driven Failures)**: Marked by gradual buildup in pressure and vibration levels with sustained elevation. These failures could benefit from **trend-based monitoring** that tracks progressive deterioration over multiple cycles.


The same logic could be used to analyze patterns on equipment setups. Below the analyzis main goal is to identify the combinations of **Preset_1** and **Preset_2** that are most frequent in operation cycles before failures.

In [ ]:
# Create a DataFrame to hold pre-failure data
presets_pre_failure = pd.DataFrame()

# Loop through each failure event to extract pre-failure data
for event in failure_events['events']:
    start_cycle = event[0]
    print(f"Failure event from cycle {start_cycle} to {event[1]}")

    # Get buffer of two operations cycles before beggining of failures
    buffer_data = get_buffer_around_index(
                df=df,
                start_index=start_cycle,
                stop_index=start_cycle,  # Only need start point
                buffer_before=2,
                buffer_after=1,  # buffer = 1 to consider failure start point
                exclude_failure_cycles=True  # Exclude cycles already in failure
            )
    
    # Filter columns to keep only integer types
    buffer_data = buffer_data.select_dtypes(int)

    # Add failure index to identify the failure event
    buffer_data['Fail_index'] = failure_events['events'].index(event) + 1  # Assigning failure index
    
    # Concatenate the buffer data to the main DataFrame
    presets_pre_failure = pd.concat([presets_pre_failure, buffer_data])

display(presets_pre_failure)

In [ ]:
# Heatmap plotting preset combinations on operation cycles before failure events
fig, ax = plt.subplots(figsize=(10, 5))

combo_heatmap = presets_pre_failure.groupby(['Preset_1','Preset_2']).size().unstack(fill_value=0)

sns.heatmap(combo_heatmap, annot=True, cmap='Reds', fmt='d')
plt.title('Preset Combinations per Failure Event')
plt.show()

- Preset_2=5 appears to be a high-risk precursor. It's the most common setting used in the cycles immediately before failures occur;

- When Preset_1=2 is used (especially with Preset_2=5), it strongly correlates with impending failure

### Key Findings of Section 5.5: Pre-failure event analysis

**Early Warning Signals:**
- Nearly all 10 failures showed sensor changes 2-3 cycles before failing
- Two distinct failure patterns were identified

**Temperature Failures (Events 2, 4, 10):**
- Sudden temperature spikes before failure
- **Monitor**: Set alerts for temperature volatility

**Pressure/Vibration Failures (Events 1, 3, 5, 6, 7, 8, 9):**
- Gradual pressure and vibration increases
- **Monitor**: Track trends over multiple cycles  

**High-Risk Configuration:**
- **(Preset_1=2, Preset_2=5)**: Used before 6 failures - highest risk combination
- **Preset_1=2**: High predominance, risky configuration

# 6 EDA Conclusion

This exploratory data analysis revealed key insights into equipment failures and operational patterns:

## Key Findings Summary

### Sensor Behavior During Failures
- **Primary Indicators:** `Pressure` (+40 units) and `VibrationY` (+54 units) show the strongest failure signals
- **Clear Separation:** Minimal overlap between normal and failure distributions across all sensors
- **Early Warning:** Sensor changes detectable 2-3 cycles before failures

### Equipment Configuration Risk
- **High-Risk Combinations:** 
  - `Preset_1=1, Preset_2=5`: 16% failure rate (double the baseline)
  - `Preset_1=2, Preset_2=5`: Most common before failures
- **Safe Configuration:** `Preset_1=3, Preset_2=4` shows 0% failure rate

### Failure Patterns
- **10 Total Events:** 66 failure cycles across 800 operational cycles
- **Two Failure Types:**
  - **Temperature-driven:** Sudden thermal spikes (Events 2, 4, 10)
  - **Mechanical:** Gradual pressure/vibration buildup (Events 1, 3, 5-9)
- **Escalating Duration:** Recent failures last 5+ cycles vs. 1-3 cycles for early events

# 7. Modelling

In [130]:
from src.config import ApplicationConfig
df = pd.read_csv(ApplicationConfig().processed_data_path, index_col='Cycle')

## 7.1 Pre-processing

### 7.1.1 One-hot encoding

In [131]:
def map_bool_to_int(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """
    Maps boolean values in the specified column to integers: True → 1, False → 0.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column (str): The name of the column to convert.

    Returns:
        pd.DataFrame: Updated DataFrame with the column converted to integers.
    """
    df[column] = df[column].astype(int)
    return df

def one_hot_encode_column(df: pd.DataFrame, column: str, prefix: str = None, drop_original: bool = True) -> pd.DataFrame:
    """
    Applies one-hot encoding to a single column in the DataFrame.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column (str): The name of the column to encode.
        prefix (str, optional): Prefix for new columns. Defaults to column name.
        drop_original (bool): Whether to drop the original column. Defaults to True.

    Returns:
        pd.DataFrame: DataFrame with one-hot encoded columns.
    """
    if prefix is None:
        prefix = column

    dummies = pd.get_dummies(df[column], prefix=prefix)

    df_encoded = pd.concat([df, dummies], axis=1)

    if drop_original:
        df_encoded = df_encoded.drop(columns=[column])

    return df_encoded

def encode_dataset(df: pd.DataFrame,  
                        one_hot_columns: List[str] = None) -> pd.DataFrame:
    """
    Encodes the DataFrame by mapping boolean columns to integers and applying one-hot encoding to specified columns.

    Args:
        df (pd.DataFrame): The input DataFrame.
        bool_columns (List[str], optional): List of boolean columns to map to integers. Defaults to None.
        one_hot_columns (List[str], optional): List of columns to apply one-hot encoding. Defaults to None.

    Returns:
        pd.DataFrame: Encoded DataFrame.
    """
    if one_hot_columns is not None:
        for col in one_hot_columns:
            df = one_hot_encode_column(df, col)

    for col in df.select_dtypes(include=['bool']).columns:
        df = map_bool_to_int(df, col)

    return df

In [132]:
df_encoded = encode_dataset(df, one_hot_columns=['Preset_1', 'Preset_2'])
display(df_encoded)

,Temperature,Pressure,VibrationX,VibrationY,VibrationZ,Frequency,Fail,Preset_1_1,Preset_1_2,Preset_1_3,Preset_2_1,Preset_2_2,Preset_2_3,Preset_2_4,Preset_2_5,Preset_2_6,Preset_2_7,Preset_2_8
Cycle,,,,,,,,,,,,,,,,,,
1,44.2352,47.6573,46.4418,64.8203,66.4545,44.4832,0,0,0,1,0,0,0,0,0,1,0,0
2,60.8072,63.1721,62.0060,80.7144,81.2464,60.2287,0,0,1,0,0,0,0,1,0,0,0,0
3,79.0275,83.0322,82.6421,98.2544,98.7852,80.9935,0,0,1,0,1,0,0,0,0,0,0,0
4,79.7162,100.5086,122.3623,121.3634,118.6525,80.3156,0,0,1,0,0,0,1,0,0,0,0,0
5,39.9891,51.7648,42.5143,61.0379,50.7165,64.2452,0,0,1,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
796,50.4695,98.2354,151.5853,99.3414,148.8385,49.8414,1,0,1,0,0,0,0,0,0,1,0,0
797,49.9853,160.4336,110.9530,160.7772,109.9176,110.9193,1,1,0,0,0,0,0,1,0,0,0,0
798,79.7773,110.5354,61.3350,149.5778,129.4638,70.8534,1,1,0,0,0,0,0,1,0,0,0,0


### 7.2.2 Train and test set creation

In [133]:
from sklearn.model_selection import train_test_split

def create_train_test_split(df: pd.DataFrame, target_col='Fail', test_size=0.2, random_state=42, stratify=True):
    """
    Splits the encoded DataFrame into train and test sets.

    Args:
        df (pd.DataFrame): The DataFrame.
        target_col (str): Name of the target column.
        test_size (float): Proportion of test set.
        random_state (int): Random seed.
        stratify (bool): Whether to stratify by target.

    Returns:
        X_train, X_test, y_train, y_test (pd.DataFrame, pd.DataFrame, pd.Series, pd.Series)
    """

    X = df.drop(columns=[target_col])
    y = df[target_col]
    stratify_y = y if stratify else None

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=stratify_y
    )
    return X_train, X_test, y_train, y_test

# Usage:
X_train, X_test, y_train, y_test = create_train_test_split(df_encoded)

### 7.2.3 Normalization

In [134]:
from sklearn.preprocessing import StandardScaler

def scale_features(X_train, X_test, scaler=None):
    """
    Scales train and test features using the provided scaler (StandardScaler by default).

    Args:
        X_train (np.ndarray or pd.DataFrame): Training features.
        X_test (np.ndarray or pd.DataFrame): Test features.
        scaler: Scaler instance (if None, uses StandardScaler).

    Returns:
        X_train_scaled, X_test_scaled
    """
    if scaler is None:
        scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

# Example usage:
X_train_scaled, X_test_scaled = scale_features(X_train, X_test)

## 7.4 Baseline model (Random Forest)

- Only one-hot encoding and normalization in the existing variables
- No oversampling

In [135]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(class_weight='balanced', n_estimators=100,
                            random_state=42, min_samples_split=2)
rf.fit(X_train_scaled, y_train)
predictions = rf.predict(X_test_scaled)

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.96      0.99      0.97       147
           1       0.78      0.54      0.64        13

    accuracy                           0.95       160
   macro avg       0.87      0.76      0.80       160
weighted avg       0.95      0.95      0.95       160



- Failures (1) -> rare events, but need to be identified -> class with bad recall and precision
- Best to improve high recall, minimize false negatives

## 7.5 Feature Engineering

### 7.5.1 Rolling window features

- Trends of increase before failures
- Capture sudden variabilities
- 3 and 5 operation cycles, estimated impact on failures

In [136]:
def add_sensor_rolling_features(df: pd.DataFrame, sensor_cols: List[str], windows: List[int] = [3, 5]):
    """
    Adds rolling window features (mean, std, percent change) for specified sensor columns and window sizes.

    For each column in sensor_cols and each window in windows, the following features are added:
        - Rolling mean: {col}_rolling_mean_{window}
        - Rolling std: {col}_rolling_std_{window}
        - Percent change over window: {col}_pct_change_{window}

    Args:
        df (pd.DataFrame): The input DataFrame.
        sensor_cols (List[str]): List of sensor column names to compute features for.
        windows (List[int], optional): List of window sizes for rolling calculations. Defaults to [3, 5].

    Returns:
        pd.DataFrame: DataFrame with new rolling window features added.
    """
    # Create a copy of the DataFrame to avoid modifying the original
    window_features = df.copy()
    
    for window in windows:
        for col in sensor_cols:
            # Rolling mean
            window_features[f'{col}_rolling_mean_{window}'] = window_features[col].rolling(window).mean()
            # Rolling standard deviation
            window_features[f'{col}_rolling_std_{window}'] = window_features[col].rolling(window).std()
            # Percent change over the window
            window_features[f'{col}_pct_change_{window}'] = window_features[col].pct_change(periods=window)
            
    return window_features

In [137]:
# Adding rolling features to the DataFrame
window_features = add_sensor_rolling_features(df_encoded, ApplicationConfig().sensor_columns, windows=[3,5])
window_features.fillna(0, inplace=True)

In [138]:
# New train-test split with rolling features
X_train, X_test, y_train, y_test = create_train_test_split(window_features)
X_train_scaled, X_test_scaled = scale_features(X_train, X_test)

# RF model with rolling features and hyperparameters
rf = RandomForestClassifier(class_weight='balanced', n_estimators=160,
                            random_state=42)

rf.fit(X_train_scaled, y_train)
pred = rf.predict(X_test_scaled)
print(f"\nRandom Forest Results:")
print(classification_report(y_test, pred))


Random Forest Results:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       147
           1       1.00      0.69      0.82        13

    accuracy                           0.97       160
   macro avg       0.99      0.85      0.90       160
weighted avg       0.98      0.97      0.97       160



- Precision 100% = 0 False Positives, no false alarms;
- Recall still needs improvement, failures were not predicted

### 7.5.2 Sensor threshold features

- failures -> related to higher values in variables

In [139]:
def add_sensor_threshold_features(df: pd.DataFrame, sensor_cols: List[str]) -> pd.DataFrame:
    """Add threshold-based alert features for sensors."""
    # Create a copy of the DataFrame to avoid modifying the original
    df_threshold = df.copy()

    for col in sensor_cols:
        # High alert: above 90th percentile
        threshold_high = df[col].quantile(0.9)
        df_threshold[f'{col}_high_alert'] = (df_threshold[col] > threshold_high).astype(int)
        
        # Extreme alert: above 95th percentile
        threshold_extreme = df[col].quantile(0.95)
        df_threshold[f'{col}_extreme_alert'] = (df_threshold[col] > threshold_extreme).astype(int)

    return df_threshold

In [140]:
# Adding threshold features to the DataFrame
df_threshold = add_sensor_threshold_features(window_features, ApplicationConfig().sensor_columns)

In [141]:
# New train-test split with rolling features
X_train, X_test, y_train, y_test = create_train_test_split(df_threshold)
X_train_scaled, X_test_scaled = scale_features(X_train, X_test)

# RF model with rolling features and hyperparameters
rf = RandomForestClassifier(class_weight='balanced', n_estimators=180, min_samples_leaf=2,
                            random_state=42)

rf.fit(X_train_scaled, y_train)
pred = rf.predict(X_test_scaled)
print(f"\nRandom Forest Results:")
print(classification_report(y_test, pred))


Random Forest Results:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       147
           1       1.00      0.85      0.92        13

    accuracy                           0.99       160
   macro avg       0.99      0.92      0.95       160
weighted avg       0.99      0.99      0.99       160



- Slightly better, 0.85 recall
- Create features for Preset_1 and Preset_2

### 7.5.3 Setup combinations feature

In [143]:
def add_high_risk_preset_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add binary features for high-risk preset combinations."""
    
    # High-risk combinations during failures
    df['high_risk_combo_1_5'] = ((df['Preset_1'] == 1) & (df['Preset_2'] == 5)).astype(int)
    df['high_risk_combo_3_5'] = ((df['Preset_1'] == 3) & (df['Preset_2'] == 5)).astype(int)
    df['high_risk_combo_1_2'] = ((df['Preset_1'] == 1) & (df['Preset_2'] == 2)).astype(int)

    return df

In [144]:
df = add_high_risk_preset_features(df)
df_encoded = encode_dataset(df, one_hot_columns=ApplicationConfig().categorical_columns)
window_features = add_sensor_rolling_features(df_encoded, ApplicationConfig().sensor_columns, windows=[3,5])
window_features.fillna(0, inplace=True)
df_threshold = add_sensor_threshold_features(window_features, ApplicationConfig().sensor_columns)

In [150]:
# New train-test split with rolling features and high-risk preset features
X_train, X_test, y_train, y_test = create_train_test_split(df_threshold)
X_train_scaled, X_test_scaled = scale_features(X_train, X_test)

# RF model with rolling features and hyperparameters
rf = RandomForestClassifier(class_weight='balanced', n_estimators=200, min_samples_leaf=2,
                            random_state=42)

rf.fit(X_train_scaled, y_train)
pred = rf.predict(X_test_scaled)
print(f"\nRandom Forest Results:")
print(classification_report(y_test, pred))


Random Forest Results:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       147
           1       1.00      0.85      0.92        13

    accuracy                           0.99       160
   macro avg       0.99      0.92      0.95       160
weighted avg       0.99      0.99      0.99       160



- Hyperparemeters improved, add features for preset (risk-based features)

- use smote to increase samples
- use cross-validation to certify that results are good with different sets

## 7.6 Oversampling with SMOTE

- few samples
- oversampling helps minimize this
- Cross validation to assure good results in different scenarios

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import confusion_matrix


# Define model pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(random_state=42,
                                     n_estimators=150,
                                     min_samples_leaf=5))
])

pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)

print(f"\nRandom Forest Results:")
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))


Random Forest Results:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99       147
           1       0.80      0.92      0.86        13

    accuracy                           0.97       160
   macro avg       0.90      0.95      0.92       160
weighted avg       0.98      0.97      0.98       160

[[144   3]
 [  1  12]]


- Improved, nice recall. Precision lower
- Test different models

## 7.7 Testing different models

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer, recall_score, precision_score, f1_score, accuracy_score
import pandas as pd
import numpy as np

# Define models to test
models = {
    'Random Forest': RandomForestClassifier(
        random_state=42,
        n_estimators=150,
        min_samples_leaf=5,
        max_features='sqrt',
    ),
    'Gradient Boosting': GradientBoostingClassifier(
         random_state=42,
        n_estimators=200,            # More estimators can improve learning
        learning_rate=0.05,          # Lower learning rate with more estimators can improve generalization
        max_depth=3,
        subsample=0.8,               # Stochastic GB to reduce overfitting
        min_samples_leaf=3
    ),
    'XGBoost': XGBClassifier(
        random_state=42,
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,                 # Sometimes a bit deeper helps
        subsample=0.8,              # For stochasticity, helps with overfitting
        colsample_bytree=0.8,       # Controls feature subsampling
        eval_metric='logloss',
        use_label_encoder=False      # Avoid warning with recent xgboost versions
    )
}

# Define scoring metrics
scoring = {
    'recall': make_scorer(recall_score),
    'precision': make_scorer(precision_score),
    'f1': make_scorer(f1_score),
    'accuracy': make_scorer(accuracy_score)
}

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store results for comparison
results_comparison = {}

print("=== MODEL COMPARISON ===\n")

for model_name, model in models.items():
    print(f"Testing {model_name}...")
    
    # Create pipeline with SMOTE
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])
    
    # Cross-validation
    cv_scores = cross_validate(
        pipeline, X_train, y_train, 
        cv=cv, scoring=scoring, 
        return_train_score=False
    )
    
    # Calculate mean and std for each metric
    results = {}
    for metric in ['test_recall', 'test_precision', 'test_f1', 'test_accuracy']:
        results[metric.replace('test_', '')] = {
            'mean': cv_scores[metric].mean(),
            'std': cv_scores[metric].std()
        }
    
    results_comparison[model_name] = results
    
    # Train on full training set and test
    pipeline.fit(X_train, y_train)
    y_pred_test = pipeline.predict(X_test)
    
    print(f"\n{model_name} - Test Set Results:")
    print(classification_report(y_test, y_pred_test))
    
    # Confusion matrix with detailed analysis
    cm = confusion_matrix(y_test, y_pred_test)
    tn, fp, fn, tp = cm.ravel()
    
    miss_rate = fn / (fn + tp) if (fn + tp) > 0 else 0
    false_alarm_rate = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    print(f"Confusion Matrix: {cm.tolist()}")
    print(f"Miss Rate (Critical): {miss_rate:.1%}")
    print(f"False Alarm Rate: {false_alarm_rate:.1%}")
    print(f"True Positives: {tp}, False Negatives: {fn}")
    print("-" * 50)

# Create comparison summary
print("\n=== CROSS-VALIDATION SUMMARY ===")
comparison_df = pd.DataFrame()

for model_name, metrics in results_comparison.items():
    model_results = {}
    for metric, values in metrics.items():
        model_results[f'{metric}_mean'] = values['mean']
        model_results[f'{metric}_std'] = values['std']
    
    comparison_df[model_name] = model_results

comparison_df = comparison_df.T.round(4)
print(comparison_df)

# Highlight best performing model for each metric
print("\n=== BEST PERFORMERS ===")
for metric in ['recall', 'precision', 'f1', 'accuracy']:
    best_model = comparison_df[f'{metric}_mean'].idxmax()
    best_score = comparison_df.loc[best_model, f'{metric}_mean']
    print(f"Best {metric.title()}: {best_model} ({best_score:.4f})")

=== MODEL COMPARISON ===

Testing Random Forest...

Random Forest - Test Set Results:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99       147
           1       0.80      0.92      0.86        13

    accuracy                           0.97       160
   macro avg       0.90      0.95      0.92       160
weighted avg       0.98      0.97      0.98       160

Confusion Matrix: [[144, 3], [1, 12]]
Miss Rate (Critical): 7.7%
False Alarm Rate: 2.0%
True Positives: 12, False Negatives: 1
--------------------------------------------------
Testing Gradient Boosting...

Gradient Boosting - Test Set Results:
              precision    recall  f1-score   support

           0       0.99      0.97      0.98       147
           1       0.75      0.92      0.83        13

    accuracy                           0.97       160
   macro avg       0.87      0.95      0.91       160
weighted avg       0.97      0.97      0.97       160

Confusion Matr

## 7.8 Variable importance

In [ ]:
X_reduced = X_train.drop(columns=['cycles_since_last_failure'])

pipeline.fit(X_reduced, y_train)
y_pred = pipeline.predict(X_reduced)

from sklearn.metrics import classification_report
print(classification_report(y_train, y_pred))


In [ ]:
# Get feature names (after scaling, so same as input)
X_reduced = X_train.drop(columns=['cycles_since_last_failure'])
feature_names = X_reduced.columns

pipeline.fit(X_reduced, y_train)

# Get importances from final model in pipeline
importances = pipeline.named_steps['model'].feature_importances_

# Create a DataFrame
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

# Plot top N features
top_n = 20
plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=importance_df.head(top_n))
plt.title(f'Top {top_n} Most Important Features')
plt.tight_layout()
plt.show()

y_pred = pipeline.predict(X_reduced)

from sklearn.metrics import classification_report
print(classification_report(y_train, y_pred))

---

## Possible Feature Engineering Strategy

### Sensor Features
- Rolling statistics (3-cycle windows) for `Pressure`, `VibrationY`, `Temperature`
- Change detection and threshold indicators

### Configuration Features
- High-risk combination flags

### Temporal Features
- Time since last failure

---

## Next Steps

1. Prepare and label datasets with engineered features  
2. Address class imbalance and define evaluation metrics  
3. Train and validate baseline models (e.g., Random Forest, XGBoost)  
4. Interpret feature importance and refine input space 